<a href="https://colab.research.google.com/github/hcopley/st_542_ml_systematic_review/blob/main/SuperLearner_in_parallel_with_folds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:

install.packages(
c('SuperLearner'
,'doParallel'
,'arm')
)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘lme4’, ‘abind’, ‘coda’




In [28]:
# Load all required libraries after installation
library(tidyverse)
library(SuperLearner)
library(data.table)
library(parallel)
library(doParallel)
library(arm)

Loading required package: MASS


Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select


Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loading required package: lme4


arm (Version 1.15-2, built: 2026-3-22)


Working directory is /content




In [19]:
dt_tfidf <- readRDS("tfidf_svd_features.rds")
dt_doc2vec <- readRDS('doc2vec_features.rds' )
dt_pubmed <- readRDS('pubmedbert_embeddings.rds')

#load in the folds created in the data preparation step.
cv_folds <- readRDS("cv_folds.rds")

In [29]:
superlearner_parallel_pipeline <- function(.data, cv_folds, run_test_sample = FALSE, test_sample_n_cols = 25) {

  #stop the previous cluster just in case it is still running from a previous run of the notebook.
  #This prevents errors about clusters already running and ensures a clean start for parallel processing.
 if(exists("cl")) {
  suppressWarnings(stopCluster(cl))
 }

  # Separate the response from features
  Y <- .data[[2]]
  X <- .data[, -c(1, 2)]

  if(run_test_sample ==TRUE) {
    # For quick testing, use a smaller sample of columns and rows. This is not for final results but can speed up iterations during development.
    set.seed(542)
    sample_columns <- sample(ncol(X), min(test_sample_n_cols, ncol(X)))

    X <- X[, sample_columns]
  }

  # Ensure that the response is mumeric
  Y <- as.numeric(Y)

  # Compute class weights
  pos_rate <- mean(Y == 1)
  neg_rate <- mean(Y == 0)

  # Assign inverse class frequency weights to balance the classes
  obs_weights <- ifelse(Y == 1, 1/pos_rate, 1/neg_rate)

  #set the number of cores for parallel processing. We use one less than the total to avoid overloading the system.
cl <- makeCluster(detectCores() - 1)
#set up parallel backend to use the cluster
registerDoParallel(cl)
#set a seed on each worker to ensure reproducibility across parallel processes
clusterSetRNGStream(cl, iseed = 42)
#send SuperLearner to the cluster workers so they can run the learners in parallel
clusterEvalQ(cl, {library(SuperLearner)})
#send all necessary objects to each cluster worker so they can run the learners in parallel without errors
#the envir = environment() argument ensures that the objects are taken from the current environment of the function
#which is important to avoid issues with object not found errors in the workers as the default is for clusterExport to look in the global environment which may not have the necessary objects when running inside a function.xpor
clusterExport(cl, list(
  "Y", "X", "learners", "obs_weights", "cv_folds"
), envir = environment())

sl_cv <- CV.SuperLearner(
  Y = Y,
  X = X,
  SL.library = learners,
  family = binomial(),
  obsWeights = obs_weights,
  method = "method.NNLS",
  cvControl = list(V = 5, validRows = cv_folds), # when passing the folds set V here to the number of folds expected
  parallel = cl
)

stopCluster(cl)

return(sl_cv)

}

In [30]:
learners <- c(
  "SL.glmnet",        # Elastic-net logistic (BEST for TF-IDF)
  "SL.xgboost",       # Shallow boosted trees
 "SL.ranger",
 "SL.ksvm",
 "SL.bayesglm"
)

In [26]:
pubmed_res <- superlearner_parallel_pipeline(dt_pubmed, cv_folds, run_test_sample = TRUE, test_sample_n_cols = 10)

ERROR: Error in checkForRemoteErrors(val): 5 nodes produced errors; first error: You have selected bayesglm as a library algorithm but either do not have the arm package installed or it can not be loaded
